In [11]:
"""
Feature Group 检查器
====================
查看每个组有哪些特征，并标注潜在leakage风险
"""

import pandas as pd
import numpy as np
import json

GROUPS_FILE = 'data/feature_groups_CLEAN.json'
DATA_FILE   = 'data/merged_data_with_momentum_CLEAN.parquet'
TARGET      = 'EWS_15min'

# 已知leakage特征
KNOWN_LEAKAGE = {
    'is_episode_t', 'is_luld', 'is_anomaly',
    'is_price_spike', 'is_volume_surge', 'thread_concentration'
}

with open(GROUPS_FILE, 'r') as f:
    feature_groups = json.load(f)

df = pd.read_parquet(DATA_FILE)
df['timestamp'] = pd.to_datetime(df['timestamp'])

print("=" * 70)
print(f"  Feature Group 检查报告")
print(f"  数据维度: {df.shape}  |  目标: {TARGET}")
print("=" * 70)

for group, feats in feature_groups.items():
    exist   = [f for f in feats if f in df.columns]
    missing = [f for f in feats if f not in df.columns]

    print(f"\n【{group.upper()}】  ({len(exist)} 个特征)")
    print("-" * 70)
    print(f"  {'特征名':<35} {'与EWS相关性':>10}  {'状态'}")
    print("-" * 70)

    for f in exist:
        corr = df[f].corr(df[TARGET])
        
        # 标注状态
        if f in KNOWN_LEAKAGE:
            status = "🔴 LEAKAGE 已知"
        elif abs(corr) > 0.4:
            status = "🟠 相关性极高，检查"
        elif abs(corr) > 0.2:
            status = "🟡 相关性偏高，留意"
        elif df[f].nunique() <= 1:
            status = "⚫ 常数列，无用"
        elif df[f].isna().mean() > 0.5:
            status = "⚪ 缺失>50%"
        else:
            status = "✅ 正常"

        corr_str = f"{corr:>+.3f}" if not np.isnan(corr) else "   NaN"
        print(f"  {f:<35} {corr_str:>10}  {status}")

    if missing:
        print(f"\n  ⚠️  以下特征在数据中不存在: {missing}")

print("\n" + "=" * 70)
print(f"  总特征数: {sum(len(v) for v in feature_groups.values())}")
print(f"  已知leakage: {KNOWN_LEAKAGE}")
print("=" * 70)

  Feature Group 检查报告
  数据维度: (59146, 120)  |  目标: EWS_15min

【MARKET】  (36 个特征)
----------------------------------------------------------------------
  特征名                                    与EWS相关性  状态
----------------------------------------------------------------------
  close                                   +0.098  ✅ 正常
  high                                    +0.101  ✅ 正常
  low                                     +0.095  ✅ 正常
  trade_count                             +0.165  ✅ 正常
  open                                    +0.097  ✅ 正常
  vwap                                    +0.098  ✅ 正常
  returns                                 +0.045  ✅ 正常
  returns_mean                            +0.054  ✅ 正常
  returns_std                             +0.142  ✅ 正常
  returns_z                               +0.021  ✅ 正常
  is_luld                                 +0.211  🔴 LEAKAGE 已知
  is_price_spike                          +0.220  🔴 LEAKAGE 已知
  is_anomaly                              +0.472 